In [ ]:
res = {}
tickers = ['AAPL','AMD','CSCO','GOOG','INTC','JNPR','META','MSFT','NFLX','NVDA','TSLA']
#tickers = ['AAPL','AMD']
res_table = pd.DataFrame(columns=tickers, index=tickers)

for s1 in tickers:
    for s2 in tickers:
        
        if s1 == s2:
            continue
        
        ####SETUP####
        #Portfolio
        start_money, max_trade_perc = 1000, 0.05
        track_value = False
        portfolio = {'money':start_money, 'max_trade':start_money*max_trade_perc, 'stocks':{}, 'b_count':0, 's_count':0, 'val_hist':[start_money]}
        

        ###Strategy
        stock_1, stock_2 = master.close[s1], master.close[s2]
        spread = stock_1 - stock_2
        
        spread_mavg5 = spread.rolling(window=5, center=False).mean()
        spread_mavg20 = spread.rolling(window=20, center=False).mean()
        std_20 = spread.rolling(window=20, center=False).std()
        
        zscore_20_5 = (spread_mavg5 - spread_mavg20)/std_20
        rolling_momentum = zscore_20_5.diff().rolling(window=3, center=False).mean()
        
        p_score = np.NaN
        for date, score in zscore_20_5.items():   
            momentum = score - p_score
            m_th = 0.1
            if (score>-1) & (momentum>m_th):
                signal = 'buy'
            elif (score<1) & (momentum<-m_th):
                signal = 'sell'
            else:
                signal = None
                
            if signal:
                order = [s1, stock_1.loc[date] , signal]
                portfolio = trade(order, portfolio)
            p_score = score
        
            if track_value: 
                portfolio['val_hist'].append(portfolio_value(portfolio, master.loc[date].close)['Total'])
            
        res[f'{s1},{s2}'] = (portfolio, portfolio_value(portfolio, master.loc[date].close))  
        res_table.loc[s1,s2] = res[f'{s1},{s2}'][1]['Total']


res_table = res_table.astype('float')
plt.figure(figsize=(15,10))
sns.heatmap(res_table, annot=True, fmt=".2f")
plt.show()

In [ ]:

if track_value:
    portfolio['val_hist'].append(portfolio_value(portfolio, master.loc[date].close)['Total'])

    
if track_value:
    fig, axes = plt.subplots(nrows=ceil(len(pairs),2), ncols=2, figsize=(15,25) )
    i, j = -1, True
    
    for s1,s2 in pairs:
        
        if j == True:
            i +=1
        j = not(j)
        
            
        stock_combo = f'{s1},{s2}'
        axes[i][int(j)].plot(res[stock_combo][0]['val_hist'], label = stock_combo)
        axes[i][int(j)].legend(loc='lower left')
    
        
    plt.show()        